# RespiScan Experimentation: ResNet18 vs MobileNetV2 Evaluation
Notebook ini berisi eksperimen komparatif awal untuk menguji kinerja arsitektur **ResNet18** dan **MobileNetV2** pada dataset audio batuk TBC. Notebook ini telah dibersihkan dari output visual lama agar file berukuran kecil (<100KB) dan dapat dimuat secara instan.

### 1. Inisialisasi & Setup Dataloader Global
Mengimpor pustaka utama PyTorch dan torchvision untuk perbandingan model.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = 'RespiScan_Train_Ready'

clean_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x.upper()), clean_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=0) for x in ['train', 'val']}

print(f"🚀 Dataloader komparatif siap di perangkat: {DEVICE}")

### 2. Modifikasi Komparatif ResNet18
Membangun arsitektur ResNet18 dengan Dense layer fully-connected termodifikasi dan pengoptimal Adam.

In [ ]:
# 1. Load Pretrained ResNet18
model_resnet = models.resnet18(weights='DEFAULT')

for param in model_resnet.parameters():
    param.requires_grad = True

num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)
)
model_resnet = model_resnet.to(DEVICE)

print("✅ Model ResNet18 untuk klasifikasi patologis siap.")

### 3. Modifikasi Komparatif MobileNetV2
Membangun MobileNetV2 yang ringan untuk perbandingan performa latensi dan akurasi.

In [ ]:
# 1. Load Pretrained MobileNetV2
model_mobilenet = models.mobilenet_v2(weights='DEFAULT')

for param in model_mobilenet.parameters():
    param.requires_grad = True

num_ftrs = model_mobilenet.classifier[1].in_features
model_mobilenet.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)
)
model_mobilenet = model_mobilenet.to(DEVICE)

print("✅ Model MobileNetV2 untuk klasifikasi patologis siap.")